# 04 -- Model

`qwenmoe/model/` is the biggest subpackage: normalization, three
positional-encoding strategies, two attention variants (GQA baseline + MLA
stretch), the Mixture-of-Experts feed-forward layer (three activation
variants + two load-balancing strategies), the block/model assembly,
autoregressive generation, and the tracing utilities every visualization
builds on.

Sections below: **RMSNorm** -> **positional schemes** -> **attention** ->
**MoE** -> **block & full model** -> **generation** -> **tracing**.

*Part of the [training-my-moe code tour](00_overview.ipynb) -- see also: [config](01_config.ipynb) - [tokenizer](02_tokenizer.ipynb) - [data](03_data.ipynb) - [model](04_model.ipynb) - [train](05_train.ipynb) - [viz](06_viz.ipynb).*

In [1]:
from pathlib import Path
import os

# Walk up until we find pyproject.toml, so these cells work no matter where Jupyter's cwd starts.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
print("Working directory:", Path.cwd())

import torch
import torch.nn.functional as F
from qwenmoe.config import ModelConfig

Working directory: /Users/sjoseph/Documents/Study/training-my-moe


## RMSNorm (`rmsnorm.py`)
Root-mean-square normalization: rescales each position's vector by
$1/\sqrt{{\text{{mean}}(x^2) + \epsilon}}$, then applies a learned
per-dimension scale. Cheaper than LayerNorm (no mean-centering step) and
what Llama/Qwen-style models use. Full derivation: `journey_of_a_model/003-norm-and-positional.md`.

In [2]:
from qwenmoe.model.rmsnorm import RMSNorm

norm = RMSNorm(dim=8)
x = torch.randn(2, 5, 8) * 10  # large-magnitude input, to make the rescaling obvious

out = norm(x)
manual_rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + norm.eps)
manual_out = (x / manual_rms) * norm.weight

print("matches the manual formula exactly:", torch.allclose(out, manual_out))
print("output RMS per position (should be ~1.0, since weight starts at 1):")
print(out.pow(2).mean(dim=-1).sqrt()[0])

matches the manual formula exactly: True
output RMS per position (should be ~1.0, since weight starts at 1):
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SelectBackward0>)


## Positional schemes (`positional.py`)
All three share the same two hooks so `Attention` can treat them
uniformly: `apply_to_qk(q, k, positions)` (a rotation, or identity) and
`attention_bias(seq_len)` (an additive score bias, or `None`).

### `RotaryEmbedding` (RoPE)

In [3]:
from qwenmoe.model.positional import RotaryEmbedding

rope = RotaryEmbedding(head_dim=8, theta=10000.0)
print("inv_freq:", rope.inv_freq)
print("inv_freq[0] is always exactly 1.0:", rope.inv_freq[0].item() == 1.0)

positions = torch.arange(4)
q = torch.randn(1, 1, 4, 8)
k = torch.randn(1, 1, 4, 8)
q_rot, k_rot = rope.apply_to_qk(q, k, positions)

print("position 0 gets NO rotation (angle=0):   ", torch.allclose(q_rot[:, :, 0], q[:, :, 0]))
print("position >0 DOES get rotated:            ", not torch.allclose(q_rot[:, :, 1], q[:, :, 1]))

inv_freq: tensor([1.0000, 0.1000, 0.0100, 0.0010])
inv_freq[0] is always exactly 1.0: True
position 0 gets NO rotation (angle=0):    True
position >0 DOES get rotated:             True


### `NoPositional`

In [4]:
from qwenmoe.model.positional import NoPositional

nope = NoPositional()
q2, k2 = nope.apply_to_qk(q, k, positions)
print("NoPositional is a pure identity:", torch.equal(q2, q), torch.equal(k2, k))
print("attention_bias:", nope.attention_bias(seq_len=4))

NoPositional is a pure identity: True True
attention_bias: None


### `AlibiBias`

In [5]:
from qwenmoe.model.positional import AlibiBias

alibi = AlibiBias(n_heads=8)
print("geometric-sequence slopes:", alibi.slopes)

bias = alibi.attention_bias(seq_len=5)
print("bias shape (n_heads, seq_len, seq_len):", bias.shape)
print("head 0's bias matrix (more negative the further back a key is):")
print(bias[0])

geometric-sequence slopes: tensor([0.5000, 0.2500, 0.1250, 0.0625, 0.0312, 0.0156, 0.0078, 0.0039])
bias shape (n_heads, seq_len, seq_len): torch.Size([8, 5, 5])
head 0's bias matrix (more negative the further back a key is):
tensor([[ 0.0000,  0.5000,  1.0000,  1.5000,  2.0000],
        [-0.5000,  0.0000,  0.5000,  1.0000,  1.5000],
        [-1.0000, -0.5000,  0.0000,  0.5000,  1.0000],
        [-1.5000, -1.0000, -0.5000,  0.0000,  0.5000],
        [-2.0000, -1.5000, -1.0000, -0.5000,  0.0000]])


### `build_positional_scheme` factory

In [6]:
from qwenmoe.model.attention import build_positional_scheme

for scheme in ["rope", "nope", "alibi"]:
    cfg = ModelConfig(vocab_size=10, d_model=16, n_heads=4, positional_scheme=scheme)
    built = build_positional_scheme(cfg, head_dim=4)
    print(scheme, "->", type(built).__name__)

rope -> RotaryEmbedding
nope -> NoPositional
alibi -> AlibiBias


## Attention (`attention.py`)
`Attention` (GQA baseline) and `MLAAttention` (task 16 stretch) both
expose the identical `forward(x, positions) -> Tensor` signature, so
`TransformerBlock` never needs to know which one it holds.

### `Attention` -- scaled dot-product self-attention with GQA

In [7]:
from qwenmoe.model.attention import Attention

def tiny_config(**overrides):
    defaults = dict(vocab_size=50, d_model=32, n_heads=4, n_kv_heads=2, max_seq_len=16)
    defaults.update(overrides)
    return ModelConfig(**defaults)

attn = Attention(tiny_config())
x_attn = torch.randn(2, 6, 32)
out = attn(x_attn, torch.arange(6))
print("output shape:", out.shape, "(matches input shape, always)")
print("n_rep (query heads sharing each kv head):", attn.n_rep)

output shape: torch.Size([2, 6, 32]) (matches input shape, always)
n_rep (query heads sharing each kv head): 2


### Causal masking, demonstrated
Perturbing only the *last* token must not change any earlier position's
output -- that's what the causal mask guarantees.

In [8]:
torch.manual_seed(0)
attn2 = Attention(tiny_config(positional_scheme="nope"))
x_causal = torch.randn(1, 5, 32)
positions = torch.arange(5)

with torch.no_grad():
    out_before = attn2(x_causal, positions)
    x_perturbed = x_causal.clone()
    x_perturbed[:, -1, :] += 10.0
    out_after = attn2(x_perturbed, positions)

print("earlier tokens' outputs unchanged by perturbing the LAST token:",
      torch.allclose(out_before[:, :-1], out_after[:, :-1], atol=1e-6))
print("the last token's own output DOES change:                       ",
      not torch.allclose(out_before[:, -1], out_after[:, -1], atol=1e-6))

earlier tokens' outputs unchanged by perturbing the LAST token: True
the last token's own output DOES change:                        True


### `MLAAttention` -- simplified Multi-Head Latent Attention (task 16, stretch)
Compresses K/V into one shared low-rank latent, and carries positional
information on a small *extra* decoupled-RoPE sub-dimension rather than
rotating the content dimensions directly. Full derivation + a
hand-verified worked example: `journey_of_a_model/016-mla-attention.md`.

In [9]:
from qwenmoe.model.attention import MLAAttention

mla_config = tiny_config(attention_variant="mla", mla_latent_dim=8)
mla = MLAAttention(mla_config)
x_mla = torch.randn(2, 6, 32)
out_mla = mla(x_mla, torch.arange(6))

print("MLA output shape (same interface as Attention):", out_mla.shape)
print("head_dim:", mla.head_dim, " rope_head_dim:", mla.rope_head_dim, " latent_dim:", mla.latent_dim)

MLA output shape (same interface as Attention): torch.Size([2, 6, 32])
head_dim: 8  rope_head_dim: 4  latent_dim: 8


### MLA's whole point: a smaller KV cache
`kv_cache_elements_per_token` is what generation would actually need to
cache per token per layer -- the latent plus the one shared decoupled-RoPE
key -- compared to GQA's `2 * n_kv_heads * head_dim` (a full K and V per kv
head). Real measured numbers from this project's own presets:

In [10]:
import dataclasses
from configs.presets import SMOKE_TEST, BASELINE

for name, base in [("SMOKE_TEST", SMOKE_TEST), ("BASELINE", BASELINE)]:
    head_dim = base.d_model // base.n_heads
    mla_cfg = dataclasses.replace(base, attention_variant="mla", mla_latent_dim=head_dim)
    mla_variant = MLAAttention(mla_cfg)

    gqa_cache = 2 * base.n_kv_heads * head_dim           # K + V, each n_kv_heads * head_dim
    mla_cache = mla_variant.kv_cache_elements_per_token  # latent + shared decoupled-RoPE key
    print(f"{name:10s} GQA cache={gqa_cache:4d}  MLA cache={mla_cache:3d}  ({mla_cache / gqa_cache:.1%} of GQA)")

SMOKE_TEST GQA cache=  64  MLA cache= 24  (37.5% of GQA)
BASELINE   GQA cache= 256  MLA cache= 48  (18.8% of GQA)


### `build_attention` factory

In [11]:
from qwenmoe.model.attention import build_attention

for variant, extra in [("gqa", {}), ("mla", {"mla_latent_dim": 8})]:
    built = build_attention(tiny_config(attention_variant=variant, **extra))
    print(variant, "->", type(built).__name__)

gqa -> Attention
mla -> MLAAttention


## Mixture-of-Experts (`moe.py`)
A router picks `top_k` routed experts per token (renormalized weights),
plus always-on shared experts (unweighted); a routed expert that gets zero
tokens this batch is simply skipped. Two swappable axes: the expert's
activation function, and the load-balancing strategy.

### Expert activation variants: SwiGLU / GEGLU / GELU

In [12]:
from qwenmoe.model.moe import SwiGLUExpert, GEGLUExpert, GELUExpert, build_expert

x_expert = torch.randn(3, 16)
for cls in [SwiGLUExpert, GEGLUExpert, GELUExpert]:
    expert = cls(d_model=16, d_ff=32)
    out = expert(x_expert)
    n_matrices = sum(1 for _ in expert.parameters())
    n_params = sum(p.numel() for p in expert.parameters())
    print(f"{cls.__name__:14s} out shape={tuple(out.shape)}  weight matrices={n_matrices}  params={n_params}")

print()
for activation in ["swiglu", "geglu", "gelu"]:
    built = build_expert(ModelConfig(vocab_size=10, d_model=16, d_ff=32, activation=activation))
    print(activation, "->", type(built).__name__)

SwiGLUExpert   out shape=(3, 16)  weight matrices=3  params=1536
GEGLUExpert    out shape=(3, 16)  weight matrices=3  params=1536
GELUExpert     out shape=(3, 16)  weight matrices=2  params=1024

swiglu -> SwiGLUExpert
geglu -> GEGLUExpert
gelu -> GELUExpert


### `Router` + `select_top_k`

In [13]:
from qwenmoe.model.moe import Router, select_top_k

router = Router(d_model=16, n_routed_experts=4)
logits = router(torch.randn(5, 16))
probs = F.softmax(logits, dim=-1)
topk_indices, topk_weights = select_top_k(logits, probs, top_k=2)

print("router logits shape:", logits.shape)
print("topk_indices:")
print(topk_indices)
print("topk_weights (each row renormalized to sum to 1):", topk_weights.sum(dim=-1))

router logits shape: torch.Size([5, 4])
topk_indices:
tensor([[2, 0],
        [2, 0],
        [1, 0],
        [0, 3],
        [2, 0]])
topk_weights (each row renormalized to sum to 1): tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


### `AuxLossBalancer` -- Switch-Transformer-style load-balancing loss
A differentiable auxiliary loss that's minimized when routing is uniform
across experts -- higher when routing is skewed.

In [15]:
from qwenmoe.model.moe import AuxLossBalancer

balancer = AuxLossBalancer(n_routed_experts=4)
n_tokens = 8
balanced_probs = torch.full((n_tokens, 4), 0.25)
balanced_indices = torch.arange(n_tokens).remainder(4).unsqueeze(-1)  # round-robin: perfectly even

imbalanced_probs = torch.zeros(n_tokens, 4)
imbalanced_probs[:, 0] = 1.0                                          # everyone's router is CERTAIN it's expert 0
imbalanced_indices = torch.zeros(n_tokens, 1, dtype=torch.long)        # ... and everyone picks expert 0

print("selection_scores == raw logits (unbiased):", torch.equal(balancer.selection_scores(balanced_probs), balanced_probs))
print("aux_loss, perfectly balanced routing:  ", balancer.aux_loss(balanced_probs, balanced_indices).item())
print("aux_loss, everyone picks expert 0:     ", balancer.aux_loss(imbalanced_probs, imbalanced_indices).item())

selection_scores == raw logits (unbiased): True
aux_loss, perfectly balanced routing:   0.009999999776482582
aux_loss, everyone picks expert 0:      0.03999999910593033


### `LossFreeBalancer` -- DeepSeek-V3 auxiliary-loss-free balancing (task 15)
A non-trainable per-expert bias, added to the softmax score for *selection
only*; nudged after each step toward discouraging overloaded experts and
encouraging underloaded ones -- no gradient involved at all.

In [16]:
from qwenmoe.model.moe import LossFreeBalancer

lf_balancer = LossFreeBalancer(n_routed_experts=4, update_rate=0.1)
print("bias starts at zero:", lf_balancer.bias.tolist())

always_expert_0 = torch.tensor([40, 0, 0, 0])  # wildly overloaded expert 0, others idle
for round_num in range(1, 6):
    lf_balancer.update(always_expert_0)
    print(f"round {round_num}: bias = {lf_balancer.bias.tolist()}")

print()
print("expert 0 (overloaded) gets pushed DOWN; experts 1-3 (idle) get pushed UP -- this")
print("discourages selecting the overloaded expert again next time, with no gradient at all.")

bias starts at zero: [0.0, 0.0, 0.0, 0.0]
round 1: bias = [-0.10000000149011612, 0.10000000149011612, 0.10000000149011612, 0.10000000149011612]
round 2: bias = [-0.20000000298023224, 0.20000000298023224, 0.20000000298023224, 0.20000000298023224]
round 3: bias = [-0.30000001192092896, 0.30000001192092896, 0.30000001192092896, 0.30000001192092896]
round 4: bias = [-0.4000000059604645, 0.4000000059604645, 0.4000000059604645, 0.4000000059604645]
round 5: bias = [-0.5, 0.5, 0.5, 0.5]

expert 0 (overloaded) gets pushed DOWN; experts 1-3 (idle) get pushed UP -- this
discourages selecting the overloaded expert again next time, with no gradient at all.


### `MoELayer` -- ties routing, experts, and balancing together

In [17]:
from qwenmoe.model.moe import MoELayer

moe_config = ModelConfig(vocab_size=50, d_model=16, d_ff=32, n_routed_experts=4, n_shared_experts=1, top_k=1, max_seq_len=8)
moe = MoELayer(moe_config)
x_moe = torch.randn(1, 3, 16)  # only 3 tokens, top_k=1 -> some expert(s) very likely get zero tokens

out_moe, aux_loss = moe(x_moe)
print("output shape:", out_moe.shape, " aux_loss:", aux_loss.item())
print("tokens dispatched per routed expert this call:", moe.last_expert_counts.tolist())
print("(any 0 above means that expert was simply skipped this batch -- no error, no gradient)")

output shape: torch.Size([1, 3, 16])  aux_loss: 0.010527295060455799
tokens dispatched per routed expert this call: [0, 1, 1, 1]
(any 0 above means that expert was simply skipped this batch -- no error, no gradient)


## Putting a block together (`block.py`, `transformer.py`)

### `TransformerBlock` -- pre-norm attention + pre-norm MoE, each with a residual

In [18]:
from qwenmoe.model.block import TransformerBlock

block_config = tiny_config(d_ff=32, n_routed_experts=4, n_shared_experts=1, top_k=2)
block = TransformerBlock(block_config)
x_block = torch.randn(2, 6, block_config.d_model)
out_block, block_aux_loss = block(x_block, torch.arange(6))
print("output shape:", out_block.shape, " aux_loss finite:", torch.isfinite(block_aux_loss).item())

output shape: torch.Size([2, 6, 32])  aux_loss finite: True


### `QwenMoEModel` -- the full model

In [19]:
from qwenmoe.model.transformer import QwenMoEModel

torch.manual_seed(0)
full_model = QwenMoEModel(SMOKE_TEST)
input_ids = torch.randint(0, SMOKE_TEST.vocab_size, (2, 10))
logits, extras = full_model(input_ids)
print("logits shape:", logits.shape, " aux_loss:", extras["aux_loss"].item())
print("tied lm_head/embedding weight (default):", full_model.lm_head.weight is full_model.embedding.weight)

(logits.sum() + extras["aux_loss"]).backward()
print("embedding got a gradient:", full_model.embedding.weight.grad is not None)

logits shape: torch.Size([2, 10, 65])  aux_loss: 0.022212859243154526
tied lm_head/embedding weight (default): True
embedding got a gradient: True


## Generation (`generate.py`)
No KV cache -- every new token re-runs a full forward pass over the whole
sequence so far, matching this project's simplicity bias everywhere else.

### `filter_top_k` / `filter_top_p`

In [20]:
from qwenmoe.model.generate import filter_top_k, filter_top_p

demo_logits = torch.tensor([[1.0, 5.0, 2.0, 4.0, 0.5]])
print("original logits:     ", demo_logits)
print("filter_top_k(k=2):   ", filter_top_k(demo_logits, k=2))

demo_logits_2 = torch.tensor([[3.0, 1.5, 1.0, 0.5, -1.0]])
print()
print("softmax probs:       ", F.softmax(demo_logits_2, dim=-1))
print("filter_top_p(p=0.7): ", filter_top_p(demo_logits_2, top_p=0.7))

original logits:      tensor([[1.0000, 5.0000, 2.0000, 4.0000, 0.5000]])
filter_top_k(k=2):    tensor([[-inf, 5., -inf, 4., -inf]])

softmax probs:        tensor([[0.6855, 0.1529, 0.0928, 0.0563, 0.0126]])
filter_top_p(p=0.7):  tensor([[3.0000, 1.5000,   -inf,   -inf,   -inf]])


### Real `generate()` against the real trained checkpoint: greedy vs. nucleus sampling
A genuine, previously-documented finding (`journey_of_a_model/011-inference-generation.md`):
greedy decoding on this checkpoint gets stuck in a repetition loop (Holtzman
et al. 2019's "neural text degeneration"); nucleus sampling on the exact
same prompt does not.

In [21]:
from qwenmoe.model.generate import generate
from qwenmoe.tokenizer.base import build_tokenizer_from_meta
from qwenmoe.train.trainer import load_checkpoint

ckpt = load_checkpoint("runs/smoke_test/checkpoints/step_1000.pt")
trained_model = QwenMoEModel(ckpt["model_config"])
trained_model.load_state_dict(ckpt["model_state_dict"])
tokenizer = build_tokenizer_from_meta(ckpt["tokenizer_meta"])

torch.manual_seed(0)
greedy = generate(trained_model, tokenizer, "ROMEO:", max_new_tokens=80, temperature=0.0)
torch.manual_seed(0)
nucleus = generate(trained_model, tokenizer, "ROMEO:", max_new_tokens=80, temperature=1.0, top_p=0.9)

print("GREEDY (temperature=0):")
print(repr(greedy))
print()
print("NUCLEUS (top_p=0.9):")
print(repr(nucleus))

GREEDY (temperature=0):
'ROMEO:\nThe the the the the the the the the the the the the the the the the the the the'

NUCLEUS (top_p=0.9):
'ROMEO:\nThremh\nWhit whonlt I doce meaman youd I tostet and fue yoXOBETI\nThee hath thigh'


## Tracing for visualization (`trace.py`)
`forward_with_trace` mirrors `forward`'s exact computation, additionally
capturing every intermediate tensor a visualization needs into a
`TraceBundle` -- only ever for a single traced sequence (`batch == 1`).

In [22]:
input_ids_trace = torch.randint(0, SMOKE_TEST.vocab_size, (1, 6))
trace_logits, bundle = full_model.forward_with_trace(input_ids_trace)

print("token_ids:", bundle.token_ids)
print("embeddings shape:", tuple(bundle.embeddings.shape))
print("num LayerTrace entries (== n_layers):", len(bundle.layers))

layer0 = bundle.layers[0]
print()
print("layer 0: q", tuple(layer0.q.shape), " k", tuple(layer0.k.shape), " attn_output", tuple(layer0.attn_output.shape))
print("layer 0 expert_outputs keys (only experts with >=1 dispatched token):", list(layer0.expert_outputs.keys()))

try:
    full_model.forward_with_trace(torch.randint(0, SMOKE_TEST.vocab_size, (2, 6)))
except ValueError as e:
    print()
    print("batch>1 guard:", e)

token_ids: [39, 38, 0, 51, 23, 15]
embeddings shape: (6, 64)
num LayerTrace entries (== n_layers): 2

layer 0: q (4, 6, 16)  k (2, 6, 16)  attn_output (6, 64)
layer 0 expert_outputs keys (only experts with >=1 dispatched token): [0, 2]

batch>1 guard: forward_with_trace only supports batch size 1, got 2
